# Prompt Fundamentals

**Phase 01** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-01/02-prompt-fundamentals.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 01-02: Prompt Fundamentals
Phase 01: Prompt and Context Engineering

Demonstrates iterative prompt improvement using the 6 core principles:
task definition, role/persona, output format, examples, constraints, tone.
Also introduces PromptTemplate for reusable, parameterized prompts.
"""

import anthropic

client = anthropic.Anthropic()

### Sample data

In [ ]:
TRANSCRIPT = """
Alex: We need to fix the login bug before the demo on Friday.
Sam: I'll look into it today. Also the dashboard is slow, we should profile it.
Alex: Good idea. Can you also update the README with the new setup steps?
Sam: Sure. Who's handling the client call at 3pm tomorrow?
Alex: I'll take it. Jordan should send the invoice by end of week.
"""

### PromptTemplate: reusable, validated prompt templates

In [ ]:
class PromptTemplate:
    """
    A prompt template that validates required variables before rendering.
    Treats prompts as parameterized code artifacts, not ad-hoc strings.
    """

    def __init__(self, template: str, required_vars: list[str], name: str = ""):
        self.template = template
        self.required_vars = required_vars
        self.name = name

    def render(self, **kwargs) -> str:
        """Render the template with provided variables. Raises ValueError if any required var is missing."""
        missing = [v for v in self.required_vars if v not in kwargs]
        if missing:
            raise ValueError(
                f"Template '{self.name}': missing required variables: {missing}"
            )
        return self.template.format(**kwargs)

    def __repr__(self) -> str:
        return f"PromptTemplate(name={self.name!r}, vars={self.required_vars})"

### The 4 prompt versions: progressively better

In [ ]:
PROMPT_V0 = "Extract the action items from this meeting."

PROMPT_V1 = """Extract all action items from the meeting transcript below.
An action item is a task that a specific person agreed to do."""

PROMPT_V2 = """Extract all action items from the meeting transcript below.
An action item is a task that a specific person agreed to do.

Output as a numbered list. Each item: [PERSON]: [TASK] by [DEADLINE or 'no deadline']."""

PROMPT_V3 = """You are a project manager's assistant. Your job is to extract
clear action items from meeting transcripts.

Extract all action items from the transcript below. An action item is a
task that a specific person explicitly agreed to do.

Output as a numbered list. Each item: [PERSON]: [TASK] by [DEADLINE or 'no deadline'].
Do not include items that were discussed but not assigned.
Do not add commentary or explanation."""

PROMPT_V4_TEMPLATE = PromptTemplate(
    template="""You are a project manager's assistant. Your job is to extract
clear action items from meeting transcripts.

Extract all action items from the transcript below. An action item is a
task that a specific person explicitly agreed to do.

Output as a numbered list. Format each item as:
[PERSON]: [TASK] by [DEADLINE or 'no deadline']

Rules:
- Include only explicitly assigned tasks
- If no deadline is mentioned, write 'no deadline' - never leave the field blank
- Do not add commentary, headers, or explanation

Example output:
1. Sam: Fix the login bug by Friday
2. Alex: Schedule client kickoff call by no deadline
3. Jordan: Send the invoice by end of week

Transcript:
{transcript}

Tone: {tone}""",
    required_vars=["transcript", "tone"],
    name="action-items-v4"
)

### Helper: run a prompt and return response

In [ ]:
def run_prompt(prompt: str, model: str = "claude-3-5-haiku-20241022") -> str:
    """Run a prompt against the model. Returns the text output."""
    response = client.messages.create(
        model=model,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def run_with_transcript(prompt_prefix: str, transcript: str) -> str:
    """Run a prompt with an appended transcript."""
    full_prompt = f"{prompt_prefix}\n\n{transcript}"
    return run_prompt(full_prompt)

### Demo 1: Show variance with the bad prompt

In [ ]:
def demo_bad_prompt() -> None:
    print("=" * 60)
    print("DEMO 1: Vague prompt - run 3 times, notice variance")
    print("=" * 60)

    for i in range(3):
        result = run_with_transcript(PROMPT_V0, TRANSCRIPT)
        print(f"\nRun {i+1}:\n{result}")
        print("-" * 40)

### Demo 2: Iterative improvement

In [ ]:
def demo_iterative_improvement() -> None:
    print("\n" + "=" * 60)
    print("DEMO 2: Iterative improvement - same transcript, 4 prompts")
    print("=" * 60)

    prompts = [
        ("v0 - vague",                         PROMPT_V0),
        ("v1 - task definition",               PROMPT_V1),
        ("v2 - + output format",               PROMPT_V2),
        ("v3 - + role, constraint",            PROMPT_V3),
    ]

    for label, prompt in prompts:
        result = run_with_transcript(prompt, TRANSCRIPT)
        print(f"\n[{label}]\n{result}\n" + "-" * 40)

### Demo 3: PromptTemplate in action

In [ ]:
def demo_template() -> None:
    print("\n" + "=" * 60)
    print("DEMO 3: PromptTemplate - v4 with example and edge case handling")
    print("=" * 60)

    prompt = PROMPT_V4_TEMPLATE.render(
        transcript=TRANSCRIPT,
        tone="direct and factual"
    )
    result = run_prompt(prompt)
    print(f"\nOutput:\n{result}")

    # Show what happens with a transcript that has no deadlines
    no_deadline_transcript = """
    Jordan: Someone should look into the latency spike.
    Riley: I can do that.
    Jordan: Great. Also we should document the deployment process.
    Riley: I'll handle it.
    """
    prompt2 = PROMPT_V4_TEMPLATE.render(
        transcript=no_deadline_transcript,
        tone="direct and factual"
    )
    result2 = run_prompt(prompt2)
    print(f"\nNo-deadline transcript output:\n{result2}")

### Demo 4: Template validation

In [ ]:
def demo_template_validation() -> None:
    print("\n" + "=" * 60)
    print("DEMO 4: Template validation - missing variable raises clear error")
    print("=" * 60)

    try:
        PROMPT_V4_TEMPLATE.render(transcript=TRANSCRIPT)  # missing 'tone'
    except ValueError as e:
        print(f"Caught expected error: {e}")

    # Show a valid render
    rendered = PROMPT_V4_TEMPLATE.render(
        transcript="Short transcript.",
        tone="casual"
    )
    print(f"\nValid render (first 100 chars): {rendered[:100]}...")

### Demo 5: Consistency test - same prompt, 5 runs

In [ ]:
def demo_consistency_test() -> None:
    print("\n" + "=" * 60)
    print("DEMO 5: Consistency test - v4 prompt, 5 runs")
    print("Expected: same structured format every time")
    print("=" * 60)

    prompt = PROMPT_V4_TEMPLATE.render(
        transcript=TRANSCRIPT,
        tone="direct and factual"
    )

    for i in range(3):  # 3 runs to keep demo fast; use 5-10 in real evaluation
        result = run_prompt(prompt)
        # Simple structural check: does it start with '1.'?
        starts_with_number = result.strip().startswith("1.")
        print(f"\nRun {i+1} (starts with '1.': {starts_with_number}):\n{result}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

### Demo

In [ ]:
print("Lesson 01-02: Prompt Fundamentals")
print("Select a demo to run:\n")
print("  1. Vague prompt variance (3 runs)")
print("  2. Iterative improvement (4 prompt versions)")
print("  3. PromptTemplate in action")
print("  4. Template validation")
print("  5. Consistency test")
print("  all. Run all demos\n")

choice = input("Choice: ").strip().lower()

if choice == "1":
    demo_bad_prompt()
elif choice == "2":
    demo_iterative_improvement()
elif choice == "3":
    demo_template()
elif choice == "4":
    demo_template_validation()
elif choice == "5":
    demo_consistency_test()
elif choice == "all":
    demo_bad_prompt()
    demo_iterative_improvement()
    demo_template()
    demo_template_validation()
    demo_consistency_test()
else:
    print("Running demo 3 (PromptTemplate) as default...")
    demo_template()

---

## Self-Check

Answer these without running code first:

**1. Which principle is most clearly missing, and what is the minimal fix?**

- A. Role/Persona: add 'You are a summarization assistant.'
- B. Output Format: specify length and structure explicitly, e.g., 'Summarize in exactly 3 bullet points, one sentence each.'
- C. Constraints: add 'Do not use markdown.'
- D. Tone: specify 'Use formal language.'

**2. Which principle addresses this specific failure, and what would you add?**

- A. Task Definition: rewrite the task to be more explicit about what to extract.
- B. Examples: add an example of the correct output format.
- C. Constraints: add 'Do not add any commentary, explanation, or closing remarks.'
- D. Tone: specify 'Be direct and terse.'

**3. What architectural change would have prevented this problem?**

- A. Use a larger model that doesn't require such specific instructions.
- B. Store prompts as named constants in a central module or use a PromptTemplate class, making each prompt a single source of truth.
- C. Add version numbers to each prompt string.
- D. Use a database to store prompts instead of Python files.

**4. How do you fix this without rewriting the entire prompt?**

- A. Add more examples showing the correct output format.
- B. Add a constraint that explicitly handles the edge case: 'If no person is clearly assigned to a task, write UNASSIGNED as the person.'
- C. Switch to a more powerful model.
- D. Increase max_tokens to give the model more room to reason.

**5. What is the practical argument for including a role specification?**

- A. The API requires a role in the system prompt.
- B. A role activates relevant training patterns and sets an implicit quality bar, producing outputs that match the domain expertise implied by the role.
- C. Roles make prompts longer, which gives the model more context to work with.
- D. Without a role, the model defaults to casual language.

**6. What is the correct fix in the PromptTemplate class?**

- A. Catch the KeyError in render() and return an empty string.
- B. Validate that all required variables are present and non-empty before rendering, and raise a ValueError with a clear error message if any are missing.
- C. Make input an optional variable with a default value.
- D. Add try/except around the template.format() call.

**7. What does this ablation test tell you about this specific prompt?**

- A. Examples are the most important principle for all prompts.
- B. For this specific prompt, examples do more work than constraints. Both matter, but if you had to choose one to optimize first, it would be the examples.
- C. The prompt needs all 6 principles applied at maximum detail.
- D. An 8/10 baseline without examples means the format description is sufficient.

**8. What specific failure mode is most likely in production, and which principle would fix it?**

- A. The model will sometimes output lowercase 'contract' instead of 'CONTRACT'. Fix: add constraint 'Always use uppercase.'
- B. The model will output a valid class label but also explain its reasoning, adding extra text the downstream parser can't handle. Fix: add constraint 'Output only the class label, nothing else.'
- C. The model will refuse to classify some documents. Fix: add role 'You are a legal expert.'
- D. The model will sometimes output 'AGREEMENT' instead of 'CONTRACT'. Fix: add examples showing edge cases.

_Answers are in `checks.json` in the lesson directory._